In [12]:
import duckdb
from pathlib import Path
import arrow

from data_importers import LandRegistryImporter

In [13]:
TODAY_YEAR = arrow.now().year

In [14]:
# Load last 3 years of data up to and including the current year

lri = LandRegistryImporter(years_to_import=range(TODAY_YEAR - 2, TODAY_YEAR + 1),)
lri.import_yearly_data()

In [3]:
DUCKLAKE_FOLDER = Path("../ducklake_land_registry")
if not DUCKLAKE_FOLDER.exists():
    DUCKLAKE_FOLDER.mkdir()

In [4]:
duckdb.sql("INSTALL ducklake")

In [5]:
ducklake_metadata = DUCKLAKE_FOLDER / "metadata.ducklake_land_registry"

In [6]:
duckdb.sql(f"ATTACH 'ducklake:{ducklake_metadata}' AS ducklake_land_registry;")

In [7]:
duckdb.sql(
    f"""
    CREATE TABLE IF NOT EXISTS ducklake_land_registry.price_paid_raw AS
    SELECT
    column00 AS 'id',
    column01 AS 'price',
    column02 AS 'date',
    column03 AS 'postcode',
    column04 AS 'property_type',
    column05 AS 'old_new',
    column06 AS 'duration',
    column07 AS 'paon',
    column08 AS 'saon',
    column09 AS 'street',
    column10 AS 'locale',
    column11 AS 'town_city',
    column12 AS 'district',
    column13 AS 'county',
    column14 AS 'ppd_category',
    column15 AS 'record_type',
    FROM read_csv('{lri.DATA_FOLDER}/pp-2010.csv');
    """
)

In [ ]:
duckdb.sql(
    f"""
    SELECT * FROM ducklake_land_registry.price_paid_raw;
    """
)

In [ ]:
duckdb.sql(
    f"""
    FROM glob('{ducklake_metadata}.files/*');
    """
)

# Process monthly updates

Here the monthly updates utilises the "record_type" column where:

- a = add as a new record
- c = change / update existing record
- d = delete


In [ ]:
lri.import_monthly_update()

In [18]:
duckdb.sql(
    f"""
    DROP TABLE IF EXISTS ducklake_land_registry.monthly_updates;
    CREATE TABLE IF NOT EXISTS ducklake_land_registry.monthly_updates AS
    SELECT
    column00 AS 'id',
    column01 AS 'price',
    column02 AS 'date',
    column03 AS 'postcode',
    column04 AS 'property_type',
    column05 AS 'old_new',
    column06 AS 'duration',
    column07 AS 'paon',
    column08 AS 'saon',
    column09 AS 'street',
    column10 AS 'locale',
    column11 AS 'town_city',
    column12 AS 'district',
    column13 AS 'county',
    column14 AS 'ppd_category',
    column15 AS 'record_type',
    FROM read_csv('{lri.DATA_FOLDER}/{lri.MONTHLY_UPDATE_FILE_NAME}');
    """
)

In [19]:
duckdb.sql(
    f"""
    SELECT record_type, COUNT(*)
    FROM ducklake_land_registry.monthly_updates
    GROUP BY record_type;
    """
)

┌─────────────┬──────────────┐
│ record_type │ count_star() │
│   varchar   │    int64     │
├─────────────┼──────────────┤
│ D           │         1361 │
│ A           │        92515 │
│ C           │         2794 │
└─────────────┴──────────────┘

In [20]:
duckdb.sql(
    f"""
    INSERT INTO ducklake_land_registry.price_paid_raw
    SELECT * FROM ducklake_land_registry.monthly_updates
    WHERE record_type = 'A';
    """
)

In [21]:
duckdb.sql(
    f"""
    DELETE FROM ducklake_land_registry.price_paid_raw
    WHERE id IN (SELECT id FROM ducklake_land_registry.monthly_updates WHERE record_type = 'D');
    """
)

Unfortunately the following approach generates:
```
NotImplementedException: Not implemented Error: Complex update plans are not yet supported for updates in DuckLake
```

In [27]:
# duckdb.sql(
#     f"""
#     UPDATE ducklake_land_registry.price_paid_raw
#     SET
#         price = mu.price,
#         date = mu.date,
#         postcode = mu.postcode,
#         property_type = mu.property_type,
#         old_new = mu.old_new,
#         duration = mu.duration,
#         paon = mu.paon,
#         saon = mu.saon,
#         street = mu.street,
#         locale = mu.locale,
#         town_city = mu.town_city,
#         district = mu.district,
#         county = mu.county,
#         ppd_category = mu.ppd_category,
#         record_type = mu.record_type
#     FROM ducklake_land_registry.monthly_updates AS mu
#     WHERE ducklake_land_registry.price_paid_raw.id = mu.id AND mu.record_type = 'C';
#     """
# )

So adopt a delete and then insert approach.

In [23]:
duckdb.sql(
    f"""
    DELETE FROM ducklake_land_registry.price_paid_raw
    WHERE id IN (SELECT id FROM ducklake_land_registry.monthly_updates WHERE record_type = 'C');
    """
)

In [24]:
duckdb.sql(
    f"""
    INSERT INTO ducklake_land_registry.price_paid_raw
    SELECT * FROM ducklake_land_registry.monthly_updates
    WHERE record_type = 'C';
    """
)

In [28]:
duckdb.sql(
    f"""
    FROM ducklake_snapshots('ducklake_land_registry');
    """
)

┌─────────────┬────────────────────────────┬────────────────┬───────────────────────────────────────────────────────────────────┐
│ snapshot_id │       snapshot_time        │ schema_version │                              changes                              │
│    int64    │  timestamp with time zone  │     int64      │                      map(varchar, varchar[])                      │
├─────────────┼────────────────────────────┼────────────────┼───────────────────────────────────────────────────────────────────┤
│           0 │ 2025-05-27 19:33:01.536+00 │              0 │ {schemas_created=[main]}                                          │
│           1 │ 2025-05-27 19:33:09.627+00 │              1 │ {tables_created=[main.price_paid_raw], tables_inserted_into=[1]}  │
│           2 │ 2025-05-27 19:36:46.91+00  │              1 │ {tables_inserted_into=[1]}                                        │
│           3 │ 2025-05-27 19:41:44.832+00 │              1 │ {tables_inserted_into=[1]}  